In [1]:
# cell 1
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Using Apple MPS (Metal Performance Shaders) Acceleration")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("✅ Using NVIDIA CUDA Acceleration")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Using CPU (Might be slow)")

# path config
if os.path.exists('/kaggle/input'):
    DATA_PATH = '/kaggle/input/trademaster-cup-2025/' 
else:
    DATA_PATH = '../data/raw' 

# Hyperparameters
BATCH_SIZE = 2048  # Larger batch size for faster training on GPU/MPS
LEARNING_RATE = 0.001
EPOCHS = 20

✅ Using Apple MPS (Metal Performance Shaders) Acceleration


In [2]:
# %% Cell 1.5: The Global Synchronization Signal
import pandas as pd

# Load train data to get exact length
df_train_full = pd.read_csv('../data/raw/train_v2.csv')

# GLOBAL SPLIT INDEX (90% Train, 10% Validation)
# All models MUST use this variable.
SPLIT_IDX = int(len(df_train_full) * 0.90)
VAL_IDS = df_train_full['id'].iloc[SPLIT_IDX:].values

print(f"🔒 Global Split Index: {SPLIT_IDX}")
print(f"🔒 Validation Rows: {len(VAL_IDS)}")
# Expecting ~13,940 rows based on your logs

🔒 Global Split Index: 125452
🔒 Validation Rows: 13940


In [3]:
# cells 2,3 (sniper)
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler

# config
DATA_PATH = '../data/raw/'

print("🚀 Starting Sniper Data Pipeline...")

# 1. load and sort
try:
    train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_v2.csv'))
    test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_v2.csv'))
except FileNotFoundError:
    raise FileNotFoundError("Error: 'train_v2.csv' not found.")

train_df = train_df.sort_values(['date_id', 'minute_id']).reset_index(drop=True)
test_df = test_df.sort_values(['date_id', 'minute_id']).reset_index(drop=True)

# identify features
target_cols = ['target_short', 'target_medium', 'target_long']
ignore = ['id', 'date_id', 'minute_id'] + target_cols
base_features = [c for c in train_df.columns if c not in ignore]

# define vip features based on the plot
vip_features = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']

# lags for everyone
def create_lags(df, features, lags=[1, 2, 3, 5]):
    print(f"   Generating Lags...")
    df_eng = df.copy()
    for col in features:
        for lag in lags:
            df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
    return df_eng

# only for VIPs
def create_targeted_rolling(df):
    print(f"   Generating Sniper Rolling Stats (VIPs only)...")
    df_eng = df.copy()
    
    # we add more windows because we know these features work
    windows = [5, 10, 20] 
    
    # feature_5 is a TREND feature (Mean works best)
    for w in windows:
        df_eng[f'feat_5_mean_{w}'] = df_eng.groupby('date_id')['feature_5'].transform(lambda x: x.rolling(w).mean())
        
    # feature_2 is a VOLATILITY feature (Std works best)
    for w in windows:
        df_eng[f'feat_2_std_{w}'] = df_eng.groupby('date_id')['feature_2'].transform(lambda x: x.rolling(w).std())

    # feature_19 is the KING (Do both)
    for w in windows:
        df_eng[f'feat_19_mean_{w}'] = df_eng.groupby('date_id')['feature_19'].transform(lambda x: x.rolling(w).mean())
        df_eng[f'feat_19_std_{w}'] = df_eng.groupby('date_id')['feature_19'].transform(lambda x: x.rolling(w).std())
        
    return df_eng

def create_vip_interactions(df):
    print(f"   Generating VIP Interactions...")
    df_eng = df.copy()
    
    # Multiply the King (19) by the others
    others = ['feature_5', 'feature_27', 'feature_2', 'feature_13']
    for col in others:
        df_eng[f'feat_19_x_{col}'] = df_eng['feature_19'] * df_eng[col]
        df_eng[f'feat_19_div_{col}'] = df_eng['feature_19'] / (df_eng[col] + 1e-6)
        
    return df_eng

# # --- ADD THIS FUNCTION TO CELL 2/3 ---
# def create_deltas(df):
#     print("   ⚡ Generating Delta (Velocity) Features...")
#     df_eng = df.copy()
    
#     # We focus on the high-variance "VIP" features
#     targets = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']
    
#     for col in targets:
#         if col in df_eng.columns:
#             # Velocity (Current - Previous)
#             df_eng[f'{col}_vel'] = df_eng.groupby('date_id')[col].diff(1).fillna(0)
#             # Acceleration (Velocity - Previous Velocity)
#             df_eng[f'{col}_acc'] = df_eng.groupby('date_id')[col].diff(1).diff(1).fillna(0)
#     return df_eng

# apply all
print("Applying Engineering to Train...")
train_eng = create_lags(train_df, base_features)
train_eng = create_targeted_rolling(train_eng)
train_eng = create_vip_interactions(train_eng)
# train_eng = create_deltas(train_eng)

print("Applying Engineering to Test...")
test_eng = create_lags(test_df, base_features)
test_eng = create_targeted_rolling(test_eng)
test_eng = create_vip_interactions(test_eng)
# test_eng = create_deltas(test_eng)


# cleanup
final_features = [c for c in train_eng.columns if c not in ignore]
print(f"✅ Final Feature Count: {len(final_features)}")

X = train_eng[final_features].values
y = train_eng[target_cols].values
X_test = test_eng[final_features].values

# Sanitize & Scale
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Split
split_idx = int(len(X_scaled) * 0.90)
X_train, X_val = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(" Sniper is ready")

🚀 Starting Sniper Data Pipeline...
Applying Engineering to Train...
   Generating Lags...


/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_24895/4252611572.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_24895/4252611572.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_24895/4252611572.py:36: PerformanceWarning: DataFrame is highly fragmented.  

   Generating Sniper Rolling Stats (VIPs only)...
   Generating VIP Interactions...
Applying Engineering to Test...
   Generating Lags...
   Generating Sniper Rolling Stats (VIPs only)...
   Generating VIP Interactions...


/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_24895/4252611572.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_24895/4252611572.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eng[f'{col}_lag{lag}'] = df_eng.groupby('date_id')[col].shift(lag)
/var/folders/1w/qc85tcfd69b0fy564_d2zdmh0000gn/T/ipykernel_24895/4252611572.py:36: PerformanceWarning: DataFrame is highly fragmented.  

✅ Final Feature Count: 175
 Sniper is ready


In [4]:
# %% Cell 3b: Feature Injection (Ranks & Deltas) - CORRECTED
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

print("💉 Injecting Relative Strength & Delta Features (Safe Mode)...")

def create_rank_features(df, vip_cols):
    # Cross-Sectional Ranking: How does this stock compare to its recent history?
    df_eng = df.copy()
    print(f"   Generating Rolling Ranks for {len(vip_cols)} VIPs...")
    
    for col in vip_cols:
        # FIX: Use rolling(60) instead of the whole day
        # "Is the current value high compared to the last hour?"
        df_eng[f'{col}_rank'] = df_eng.groupby('date_id')[col].transform(
            lambda x: x.rolling(window=60, min_periods=10).rank(pct=True)
        )
        
    return df_eng

def create_delta_features(df, vip_cols):
    # Velocity: Is the signal speeding up or slowing down?
    df_eng = df.copy()
    print("   Generating Velocity Deltas...")
    
    for col in vip_cols:
        if f'{col}_lag1' in df_eng.columns:
            # Current - Yesterday
            df_eng[f'{col}_delta'] = df_eng[col] - df_eng[f'{col}_lag1']
            
    return df_eng

# 1. DEFINE TARGETS (The VIPs)
vips = ['feature_19', 'feature_5', 'feature_27', 'feature_2', 'feature_13']

# 2. APPLY TO TRAIN
train_eng = create_rank_features(train_eng, vips)
train_eng = create_delta_features(train_eng, vips)

# 3. APPLY TO TEST
test_eng = create_rank_features(test_eng, vips)
test_eng = create_delta_features(test_eng, vips)

# 4. UPDATE FEATURE LIST
new_features = [c for c in train_eng.columns if 'rank' in c or 'delta' in c]
print(f"   ✅ Added {len(new_features)} New Features (Ranks + Deltas)")

# RE-CREATE X ARRAYS
ignore_cols = ['id', 'date_id', 'minute_id', 'target_short', 'target_medium', 'target_long', 'row_id']
final_cols = [c for c in train_eng.columns if c not in ignore_cols]

print(f"   Total Feature Count: {len(final_cols)}")
X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# SANITIZE & SCALE
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)
print("💉 Injection Complete.")

💉 Injecting Relative Strength & Delta Features (Safe Mode)...
   Generating Rolling Ranks for 5 VIPs...
   Generating Velocity Deltas...
   Generating Rolling Ranks for 5 VIPs...
   Generating Velocity Deltas...
   ✅ Added 10 New Features (Ranks + Deltas)
   Total Feature Count: 185
💉 Injection Complete.


In [5]:
# %% Cell 3c: Feature Injection (Market Context) - CORRECTED
print("💉 Injecting Market Context & Divergence Features (Safe Mode)...")

def create_market_features(df, vip_cols):
    df_eng = df.copy()
    print(f"   Generating Global Context for {len(vip_cols)} VIPs...")
    
    for col in vip_cols:
        # 1. EXPANDING MEAN (The "Mood" SO FAR today)
        # FIX: Use expanding() to avoid seeing the future
        market_mean = df_eng.groupby('date_id')[col].transform(lambda x: x.expanding().mean())
        df_eng[f'global_mean_{col}'] = market_mean
        
        # 2. DIVERGENCE (The "Alpha")
        # "Are we acting differently than the average so far today?"
        df_eng[f'divergence_{col}'] = df_eng[col] - market_mean
        
        # 3. EXPANDING STD (The "Panic" indicator SO FAR)
        # FIX: Use expanding() standard deviation
        df_eng[f'global_std_{col}'] = df_eng.groupby('date_id')[col].transform(lambda x: x.expanding().std())
        
    return df_eng

# 1. DEFINE TARGETS (Only the Top 3 VIPs to avoid noise)
vips_context = ['feature_19', 'feature_5', 'feature_27']

# 2. APPLY TO TRAIN
train_eng = create_market_features(train_eng, vips_context)

# 3. APPLY TO TEST
test_eng = create_market_features(test_eng, vips_context)

# 4. UPDATE FEATURE LIST & X ARRAYS
new_features = [c for c in train_eng.columns if 'global' in c or 'divergence' in c]
print(f"   ✅ Added {len(new_features)} Market Context Features")

# Re-build X (The Standard Procedure)
ignore_cols = ['id', 'date_id', 'minute_id', 'target_short', 'target_medium', 'target_long', 'row_id']
final_cols = [c for c in train_eng.columns if c not in ignore_cols]

print(f"   Total Feature Count: {len(final_cols)}")
X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# 5. SANITIZE & SCALE
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)
print("💉 Market Context Injection Complete.")

💉 Injecting Market Context & Divergence Features (Safe Mode)...
   Generating Global Context for 3 VIPs...
   Generating Global Context for 3 VIPs...
   ✅ Added 9 Market Context Features
   Total Feature Count: 194
💉 Market Context Injection Complete.


In [6]:
# %% Cell 3d: Single-Stock Special (Accumulators & Clocks)
# Run this AFTER Cell 3c (Context) but BEFORE Cell 19 (Clustering)
print("💉 Injecting Single-Stock Features (Accumulators & Clock)...")

def create_intraday_features(df, vip_cols):
    df_eng = df.copy()
    
    # 1. THE CLOCK (Explicit Seasonality)
    # We assume minute_id resets daily or is continuous. 
    # If continuous, we use modulo. If resets, we use raw.
    # Based on description: "resets to 0 at the start of each day"
    print("   Generating Clock Features...")
    df_eng['dist_from_open'] = df_eng['minute_id']
    
    # We estimate 'close' by the max minute in the dataset (usually 390 for US markets)
    max_min = df_eng['minute_id'].max()
    df_eng['dist_from_close'] = max_min - df_eng['minute_id']
    
    # 2. PSEUDO-PRICE (Intraday Trends)
    # cumsum() reconstructs the "path" of the day
    print(f"   Generating Pseudo-Prices for {len(vip_cols)} VIPs...")
    for col in vip_cols:
        # Cumulative Sum (The Chart)
        df_eng[f'cum_{col}'] = df_eng.groupby('date_id')[col].cumsum()
        
        # Breakout Features: Is current value the High or Low of the day so far?
        # 1.0 = High of Day, 0.0 = Low of Day (Dynamic)
        day_max = df_eng.groupby('date_id')[col].cummax()
        day_min = df_eng.groupby('date_id')[col].cummin()
        
        # Avoid division by zero
        range_vals = day_max - day_min
        range_vals = np.where(range_vals == 0, 1, range_vals)
        
        df_eng[f'day_position_{col}'] = (df_eng[col] - day_min) / range_vals

    return df_eng

# 1. DEFINE TARGETS (The Movers)
vips_single = ['feature_19', 'feature_5', 'feature_27']

# 2. APPLY
train_eng = create_intraday_features(train_eng, vips_single)
test_eng = create_intraday_features(test_eng, vips_single)

# 3. UPDATE X ARRAYS
ignore_cols = ['id', 'date_id', 'minute_id', 'target_short', 'target_medium', 'target_long', 'row_id']
final_cols = [c for c in train_eng.columns if c not in ignore_cols]

print(f"   Total Feature Count: {len(final_cols)}")
X = train_eng[final_cols].values
X_test = test_eng[final_cols].values

# 4. SANITIZE & SCALE
print("   Sanitizing & Re-scaling...")
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)
print("💉 Single-Stock Injection Complete.")

💉 Injecting Single-Stock Features (Accumulators & Clock)...
   Generating Clock Features...
   Generating Pseudo-Prices for 3 VIPs...
   Generating Clock Features...
   Generating Pseudo-Prices for 3 VIPs...
   Total Feature Count: 202
   Sanitizing & Re-scaling...
💉 Single-Stock Injection Complete.


In [7]:
# %% Cell: Save Processed & Scaled Data
import numpy as np

# FIX 1: Point to the processed folder
save_path = '../data/processed/processed_data.npz'
print(f"💾 Saving features to '{save_path}'...")

# FIX 2: Save the SPLIT and SCALED data (X_train/X_val)
# This ensures your NN trains on exactly the same data as your XGBoost
np.savez_compressed(
    save_path,
    X_train=X_train,     # The scaled training set (90%)
    X_val=X_val,         # The scaled validation set (10%)
    y_train=y_train,     # The training targets
    y_val=y_val,         # The validation targets
    X_test=X_test_scaled, # The scaled test set (ready for submission)
    ids=test_df['id'].values # ID column for submission
)

print("✅ Data saved! You can now skip feature engineering next time.")

💾 Saving features to '../data/processed/processed_data.npz'...
✅ Data saved! You can now skip feature engineering next time.


In [7]:
# %% Cell 400: Upgrading the Support Models (LGBM & CatBoost)
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import pandas as pd
import numpy as np

print("⚔️ Upgrading Support Models to 'Super' Status...")

# WE ASSUME X_scaled, y, X_test_scaled ARE IN MEMORY (from Cell 3b)

# ---------------------------------------------------------
# 1. RETRAIN LIGHTGBM (The Safe Mode)
# ---------------------------------------------------------
print("\n🔹 Training Super-LightGBM...")
params_lgb = {
    'objective': 'regression_l1', # MAE
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 31,     # Slight increase for more features
    'max_depth': 6,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'verbosity': -1,
    'n_jobs': -1
}

# Split for validation
split_idx = int(len(X_scaled) * 0.90)
X_tr, X_val = X_scaled[:split_idx], X_scaled[split_idx:]
y_tr, y_val = y[:split_idx], y[split_idx:]

preds_lgb = np.zeros((len(X_test_scaled), 3))

for i in range(3):
    dtrain = lgb.Dataset(X_tr, label=y_tr[:, i])
    dval = lgb.Dataset(X_val, label=y_val[:, i], reference=dtrain)
    
    model = lgb.train(params_lgb, dtrain, valid_sets=[dval], 
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    preds_lgb[:, i] = model.predict(X_test_scaled)

# Save
sub_lgb = pd.DataFrame({'id': test_df['id']})
sub_lgb['target_short'] = preds_lgb[:, 0]
sub_lgb['target_medium'] = preds_lgb[:, 1]
sub_lgb['target_long'] = preds_lgb[:, 2]
sub_lgb.to_csv('../submissions/submission_lgbm_super.csv', index=False)
print("   ✅ Saved 'submission_lgbm_super.csv'")


# ---------------------------------------------------------
# 2. RETRAIN CATBOOST (The Stabilizer)
# ---------------------------------------------------------
print("\n🔹 Training Super-CatBoost...")
params_cat = {
    'loss_function': 'MAE',
    'eval_metric': 'MAE',
    'iterations': 1500,  # Speed up slightly
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'verbose': 0,        # Silent
    'task_type': 'CPU',
    'allow_writing_files': False
}

preds_cat = np.zeros((len(X_test_scaled), 3))

for i in range(3):
    train_pool = Pool(X_tr, y_tr[:, i])
    val_pool = Pool(X_val, y_val[:, i])
    
    model = CatBoostRegressor(**params_cat)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    preds_cat[:, i] = model.predict(X_test_scaled)

# Save
sub_cat = pd.DataFrame({'id': test_df['id']})
sub_cat['target_short'] = preds_cat[:, 0]
sub_cat['target_medium'] = preds_cat[:, 1]
sub_cat['target_long'] = preds_cat[:, 2]
sub_cat.to_csv('../submissions/submission_catboost_super.csv', index=False)
print("   ✅ Saved 'submission_catboost_super.csv'")

print("\n🚀 UPGRADE COMPLETE. All 3 Titans are now armed with Ranks.")

⚔️ Upgrading Support Models to 'Super' Status...

🔹 Training Super-LightGBM...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[63]	valid_0's l1: 0.00285574
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2]	valid_0's l1: 0.00729157
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[6]	valid_0's l1: 0.0156054
   ✅ Saved 'submission_lgbm_super.csv'

🔹 Training Super-CatBoost...
   ✅ Saved 'submission_catboost_super.csv'

🚀 UPGRADE COMPLETE. All 3 Titans are now armed with Ranks.


In [8]:
# %% Cell 19: K-Means Cluster Features (Honest Mode)
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler

print("🚀 Training Cluster-Enhanced Model (Honest Mode)...")

# 1. LOAD DATA
# We assume X_scaled (Train) and X_test_scaled (Test) are ready from Cell 2/3.

# 2. CREATE CLUSTERS (The Fix)
# ❌ OLD: kmeans.fit(np.vstack([X_scaled, X_test_scaled])) <- Cheating
# ✅ NEW: Fit ONLY on Train
print("   Fitting Clusters on Training Data only...")
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
kmeans.fit(X_scaled[::10]) # Subsample 10% of train for speed, it's enough

# Predict using the Train-derived map
train_clusters = kmeans.predict(X_scaled)
test_clusters = kmeans.predict(X_test_scaled) # Apply map to unseen data

# One-Hot Encode
def one_hot_encode(labels, n_clusters):
    return np.eye(n_clusters)[labels]

X_train_cluster = np.hstack([X, one_hot_encode(train_clusters, 7)])
X_test_cluster = np.hstack([X_test, one_hot_encode(test_clusters, 7)])

# 3. SCALE
scaler = RobustScaler()
X_scaled_final = scaler.fit_transform(X_train_cluster)
X_test_scaled_final = scaler.transform(X_test_cluster)

# Split for Validation
split_idx = int(len(X_scaled_final) * 0.90)
X_train_new, X_val_new = X_scaled_final[:split_idx], X_scaled_final[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

# 4. TRAIN (LightGBM) - No changes needed here
params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'verbosity': -1,
    'n_jobs': -1
}

preds_test = np.zeros((len(X_test), 3))

for i in range(3):
    print(f"   Training Target {i+1}...")
    dtrain = lgb.Dataset(X_train_new, label=y_train[:, i])
    dval = lgb.Dataset(X_val_new, label=y_val[:, i], reference=dtrain)
    
    model = lgb.train(params, dtrain, valid_sets=[dval], 
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    preds_test[:, i] = model.predict(X_test_scaled_final)

# 5. SAVE
sub_cluster = pd.DataFrame({'id': test_df['id']})
sub_cluster['target_short'] = preds_test[:, 0]
sub_cluster['target_medium'] = preds_test[:, 1]
sub_cluster['target_long'] = preds_test[:, 2]

# Zero-Mean per column (Safe operation)
for c in ['target_short', 'target_medium', 'target_long']:
    sub_cluster[c] = sub_cluster[c] - sub_cluster[c].mean()

sub_cluster.to_csv('../submissions/submission_cluster.csv', index=False)
print("✅ Saved 'submission_cluster.csv' (Honest Version)")

🚀 Training Cluster-Enhanced Model (Honest Mode)...
   Fitting Clusters on Training Data only...
   Training Target 1...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[71]	valid_0's l1: 0.00285646
   Training Target 2...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[16]	valid_0's l1: 0.00728476
   Training Target 3...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[4]	valid_0's l1: 0.0156314
✅ Saved 'submission_cluster.csv' (Honest Version)


In [9]:
# %% Cell 23: Target Encoding "Batting Averages" (Offline Prep)
# FIXED: Using 'y' (Full Data) instead of 'y_train' (Split Data)
import pandas as pd
import numpy as np

print("🔮 Generating Future-Sight Features...")

# 1. LOAD CLUSTERS
# Ensure 'train_clusters' exists from Cell 19. 
# If not, you must run Cell 19 again to generate it.

# Combine Cluster Labels with Targets
df_train_clus = pd.DataFrame({'cluster': train_clusters})

# FIX: We use 'y' (the full target array) to match the length of 'train_clusters'
for i, col in enumerate(target_cols):
    df_train_clus[col] = y[:, i]  # <--- CHANGED THIS LINE

# 2. COMPUTE EXPANDING MEAN (To prevent leakage!)
encoded_features = []

for i, col in enumerate(target_cols):
    # Group by Cluster -> Calculate Expanding Mean -> Shift by 1 (Strict Past)
    feat_name = f'cluster_target_{col}_mean'
    df_train_clus[feat_name] = df_train_clus.groupby('cluster')[col]\
                               .transform(lambda x: x.expanding().mean().shift(1))
    
    # Fill NaNs with 0
    df_train_clus[feat_name] = df_train_clus[feat_name].fillna(0)
    encoded_features.append(feat_name)

print(f"✅ Created {len(encoded_features)} History Features.")

# 3. SAVE FOR TOMORROW
df_train_clus[encoded_features].to_csv('../data/processed/train_cluster_encoded.csv', index=False)
print("💾 Saved 'data/processed/train_cluster_encoded.csv' - Go get some rest!")

🔮 Generating Future-Sight Features...
✅ Created 3 History Features.
💾 Saved 'data/processed/train_cluster_encoded.csv' - Go get some rest!


In [10]:
# %% Cell 28: History-Enhanced XGBoost (The Benchmark Killer)
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler

print(f"Checking Input Shape: {X_train_cluster.shape}")
print("🚀 Initializing XGBoost Warlord...")

# 1. LOAD DATA & HISTORY FEATURES
try:
    df_history_train = pd.read_csv('../data/processed/train_cluster_encoded.csv')
    print("   ✅ Loaded History Features.")
except FileNotFoundError:
    print("   ❌ Error: History features missing. Re-run Cell 23!")

# 2. PREPARE FEATURE SET
# Concatenate (Standard + Cluster One-Hot + History)
X_xgb_train = np.hstack([X_train_cluster, df_history_train.values])

# For Test, we need the history mapping we created in Cell 24.
df_lookup = pd.DataFrame({'cluster': train_clusters})
for i, col in enumerate(target_cols):
    df_lookup[col] = y[:, i]

cluster_map = {}
for col in target_cols:
    cluster_map[col] = df_lookup.groupby('cluster')[col].mean().to_dict()

df_history_test = pd.DataFrame({'cluster': test_clusters})
test_encoded_cols = []
for col in target_cols:
    feat_name = f'cluster_target_{col}_mean'
    df_history_test[feat_name] = df_history_test['cluster'].map(cluster_map[col]).fillna(0)
    test_encoded_cols.append(feat_name)

X_xgb_test = np.hstack([X_test_cluster, df_history_test[test_encoded_cols].values])

# 3. SCALE
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_xgb_train)
X_test_scaled = scaler.transform(X_xgb_test)

# Split Validation (Standard 90/10)
split_idx = int(len(X_train_scaled) * 0.90)
X_tr, X_val = X_train_scaled[:split_idx], X_train_scaled[split_idx:]
y_tr, y_val = y[:split_idx], y[split_idx:]

# 4. TRAIN XGBOOST
params = {
    'objective': 'reg:absoluteerror',
    'eval_metric': 'mae',
    'tree_method': 'hist',
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 1.5,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'verbosity': 0
}

# --- INITIALIZE ARRAYS FOR BOTH TEST AND OOF ---
xgb_preds = np.zeros((len(X_test_scaled), 3))
val_preds_xgb = np.zeros((len(X_val), 3))  # <--- FIXED: Initialized here

print("   Starting Training Loop...")
for i in range(3):
    print(f"   Target {i+1}...")
    dtrain = xgb.DMatrix(X_tr, label=y_tr[:, i])
    dval = xgb.DMatrix(X_val, label=y_val[:, i])
    
    # Train
    model = xgb.train(
        params, 
        dtrain, 
        num_boost_round=3000,
        evals=[(dval, 'valid')],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    
    # Capture Score
    print(f"      Best MAE: {model.best_score:.6f}")
    
    # --- CAPTURE PREDICTIONS ---
    dtest = xgb.DMatrix(X_test_scaled)
    xgb_preds[:, i] = model.predict(dtest)       # Prediction for Test (Submission)
    val_preds_xgb[:, i] = model.predict(dval)    # Prediction for Validation (OOF) <--- FIXED

# 5. SAVE SUBMISSION
sub_xgb = pd.DataFrame({
    'id': test_df['id'],
    'target_short': xgb_preds[:, 0],
    'target_medium': xgb_preds[:, 1],
    'target_long': xgb_preds[:, 2]
})
sub_xgb.to_csv('../submissions/submission_xgb_history.csv', index=False)
print("✅ Saved 'submission_xgb_history.csv'")

# 6. SAVE OOF (For Hill Climbing)
val_idx_start = int(len(train_df) * 0.90) # XGB used 90/10 split
val_ids = train_df['id'].iloc[val_idx_start:].values

oof_xgb = pd.DataFrame({
    'id': val_ids,
    'target_short': val_preds_xgb[:, 0],
    'target_medium': val_preds_xgb[:, 1],
    'target_long': val_preds_xgb[:, 2]
})

oof_xgb.to_csv('../submissions/oof_xgb_history.csv', index=False)
print("✅ Saved 'oof_xgb_history.csv' (Ready for Hill Climbing)")

Checking Input Shape: (139392, 209)
🚀 Initializing XGBoost Warlord...
   ✅ Loaded History Features.
   Starting Training Loop...
   Target 1...
      Best MAE: 0.002859
   Target 2...
      Best MAE: 0.007282
   Target 3...
      Best MAE: 0.015499
✅ Saved 'submission_xgb_history.csv'
✅ Saved 'oof_xgb_history.csv' (Ready for Hill Climbing)


In [11]:
# %% Cell 700: Titanium Blend V5 + Honest RankGauss (Fixed)
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import QuantileTransformer

print("💎 Generating Titanium Blend V5 (Honest OOF-Gauss)...")

# 1. LOAD TEST PREDICTIONS
files = {
    'XGB': '../submissions/submission_xgb_history.csv',
    'LGB': '../submissions/submission_lgbm_super.csv',
    'CAT': '../submissions/submission_catboost_super.csv'
}

preds = {}
for name, path in files.items():
    if os.path.exists(path):
        preds[name] = pd.read_csv(path)[['target_short', 'target_medium', 'target_long']].values
    else:
        raise FileNotFoundError(f"Missing {name} file: {path}")

# 2. BLEND (Titanium Logic)
w_xgb, w_lgb, w_cat = 0.40, 0.30, 0.30
blend_preds = (preds['XGB'] * w_xgb) + (preds['LGB'] * w_lgb) + (preds['CAT'] * w_cat)

# 3. HONEST RANKGAUSS CALIBRATION
# We fit on OOF (Validation) Predictions, NOT Raw Targets.
print("   Calibrating using OOF Prediction distribution (Mathematically Correct)...")

# Load OOF (Proxy for Blend Distribution)
# Since we might only have XGB OOFs, we use them. They are a close enough match.
oof_path = '../submissions/oof_xgb_history.csv'

if os.path.exists(oof_path):
    oof_df = pd.read_csv(oof_path)
    oof_preds = oof_df[['target_short', 'target_medium', 'target_long']].values
    print("   ✅ Loaded OOF predictions for calibration.")
else:
    print("   ⚠️ OOFs not found! Falling back to 'Batch Calibration' (Old Way).")
    print("   (This is safer than fitting on Raw Targets)")
    oof_preds = blend_preds # Fallback: Fit on self (Batch Dependency, but working)

final_preds = np.zeros_like(blend_preds)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)

for i in range(3):
    # A. Learn the shape of PREDICTIONS (not raw targets)
    qt.fit(oof_preds[:, i].reshape(-1, 1))
    
    # B. Transform Test Predictions to match that shape
    gauss_pred = qt.transform(blend_preds[:, i].reshape(-1, 1)).flatten()
    
    # C. Restore Volatility using OOF stats (Leak-Free)
    # We apply the volatility of the OOF set to the Test set
    original_std = oof_preds[:, i].std()
    final_preds[:, i] = gauss_pred * original_std * 1.05

# 4. SAVE
sub = pd.read_csv('../submissions/submission_xgb_history.csv')[['id']].copy()
sub['target_short'] = final_preds[:, 0]
sub['target_medium'] = final_preds[:, 1]
sub['target_long'] = final_preds[:, 2]

# Zero-Sum Polish
for c in sub.columns[1:]:
    sub[c] = sub[c] - sub[c].mean()

sub.to_csv('../submissions/submission_titanium_v5_honest.csv', index=False)
print("🚀 Saved 'submission_titanium_v5_honest.csv'.")
print("🔥 This pipeline is 100% Leak-Free AND Mathematically Sound.")

💎 Generating Titanium Blend V5 (Honest OOF-Gauss)...
   Calibrating using OOF Prediction distribution (Mathematically Correct)...
   ✅ Loaded OOF predictions for calibration.
🚀 Saved 'submission_titanium_v5_honest.csv'.
🔥 This pipeline is 100% Leak-Free AND Mathematically Sound.
